# 04 · Data Cleaning

**Objective:** Turn the three raw sources into one clean, merged fact table (`fact_funding_rounds_clean.csv`) and a clean unicorn dimension, using the same logic as `src/data_engineering/clean.py`, rebuilt and explained cell-by-cell here. Every decision below traces back to a specific finding from notebook 03 — nothing here is cleaning for its own sake.

**Business purpose:** a wrong join or a silently-mangled currency column doesn't crash anything — it just produces a "total funding" number that's wrong, and nobody notices until a stakeholder asks why the dashboard total doesn't match a known figure. So every step here is written to fail loudly instead (asserts, logged counts) rather than fail silently.

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

RAW_DIR = Path("data/raw")

MIN_VALID_YEAR = 1900
MAX_VALID_YEAR = 2025

# Manual overrides for country names pycountry/fuzzy matching gets wrong,
# or that lack a clean out-of-the-box ISO3 mapping.
COUNTRY_NAME_TO_ISO3_OVERRIDES = {
    "United States": "USA", "United Kingdom": "GBR", "South Korea": "KOR",
    "Vietnam": "VNM", "Russia": "RUS", "Bahamas": "BHS", "Czech Republic": "CZE",
    "Ivory Coast": "CIV", "Democratic Republic of the Congo": "COD",
    "United Arab Emirates": "ARE", "Turkey": "TUR",
}
# Known data-entry errors in the unicorn dataset's `industry` column
# (a city/generic term sitting where an industry should be).
UNICORN_INDUSTRY_FIXES = {
    "Vultr": "Enterprise Tech",
    "MindMaze": "Healthcare & Life Sciences",
}

## 1. Currency-string cleaning

From notebook 03: `funding_total_usd` is text because Crunchbase formats it with commas (`"1,750,000"`), and some rows use a lone `"-"` for zero/missing.

In [1]:
def clean_currency_string(series: pd.Series) -> pd.Series:
    """Cleans currency text like ' 17,50,000 ' or ' - ' into proper floats."""
    cleaned = series.astype(str).str.strip()
    cleaned = cleaned.replace({"-": None, "nan": None, "": None})
    cleaned = cleaned.str.replace(",", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce")

# quick unit-style check before using it on the real data
demo = pd.Series([" 1,750,000 ", " - ", "500000", None, "12,3,456"])
print("Input: ", demo.tolist())
print("Output:", clean_currency_string(demo).tolist())
print("-> confirms comma-stripping, '-' -> NaN, and whitespace handling all work as intended")

Input:  [' 1,750,000 ', ' - ', '500000', nan, '12,3,456']
Output: [1750000.0, nan, 500000.0, nan, 123456.0]
-> confirms comma-stripping, '-' -> NaN, and whitespace handling all work as intended


## 2. Impossible-date handling

Dates outside a sane range are either genuine old institutions (pre-1900) or clear typos (e.g. year 2914). I treat these **differently**: typo future-dates get nulled, while pre-1900 dates get flagged, not deleted.

In [1]:
def null_out_impossible_years(df, date_cols, min_year=MIN_VALID_YEAR, max_year=None):
    df = df.copy()
    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors="coerce")
        year = parsed.dt.year
        bad_future = year > (max_year if max_year else pd.Timestamp.now().year + 1)
        n_bad = int(bad_future.sum())
        if n_bad:
            print(f"  nulling {n_bad} impossible future-dated values in '{col}'")
        df.loc[bad_future, col] = pd.NaT
        df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

## 3. Loading + cleaning `investments_VC.csv`

In [1]:
print("Loading investments_VC.csv (latin1 encoding)")
vc = pd.read_csv(RAW_DIR / "investments_VC.csv", encoding="latin1", low_memory=False)
vc.columns = [c.strip() for c in vc.columns]

before = len(vc)
vc = vc.dropna(how="all")
print(f"  dropped {before - len(vc):,} fully-blank rows (trailing blank rows in source CSV)")

vc["permalink"] = vc["permalink"].str.lower().str.strip()
vc["funding_total_usd"] = clean_currency_string(vc["funding_total_usd"])

vc = null_out_impossible_years(vc, ["founded_at", "first_funding_at", "last_funding_at"])

round_cols = [
    "seed", "venture", "equity_crowdfunding", "undisclosed", "convertible_note",
    "debt_financing", "angel", "grant", "private_equity", "post_ipo_equity",
    "post_ipo_debt", "secondary_market", "product_crowdfunding",
    "round_A", "round_B", "round_C", "round_D", "round_E", "round_F", "round_G", "round_H",
]
for c in round_cols:
    vc[c] = pd.to_numeric(vc[c], errors="coerce").fillna(0)

print(f"\ninvestments_VC cleaned: {len(vc):,} rows retained, {vc.shape[1]} columns")

Loading investments_VC.csv (latin1 encoding)
  dropped 4,856 fully-blank rows (trailing blank rows in source CSV)

investments_VC cleaned: 49,438 rows retained, 39 columns


### Why fill round-type columns with 0, not a median/mean?
These are **structural zeros** (notebook 03, section 3): a company that never raised a Series B has `round_B = 0`, not `round_B = unknown`. Imputing with a mean here would fabricate funding activity that never happened — a textbook example of "technically valid pandas code, wrong business logic."

## 4. Loading + cleaning `big_startup_secsees_dataset.csv`

In [1]:
print("Loading big_startup_secsees_dataset.csv")
sf = pd.read_csv(RAW_DIR / "big_startup_secsees_dataset.csv", encoding="utf-8", low_memory=False)
sf["permalink"] = sf["permalink"].str.lower().str.strip()
sf["funding_total_usd"] = clean_currency_string(sf["funding_total_usd"])
sf = null_out_impossible_years(sf, ["founded_at", "first_funding_at", "last_funding_at"])

founded_year = pd.to_datetime(sf["founded_at"], errors="coerce").dt.year
sf["is_pre_1900_founding"] = founded_year < MIN_VALID_YEAR
n_flagged = int(sf["is_pre_1900_founding"].sum())
print(f"  flagged {n_flagged} rows founded before {MIN_VALID_YEAR} (KEPT, not dropped)")
print(f"\nsuccess_fail cleaned: {len(sf):,} rows retained")

Loading big_startup_secsees_dataset.csv
  nulling 4 impossible future-dated values in 'founded_at'
  nulling 2 impossible future-dated values in 'last_funding_at'
  flagged 108 rows founded before 1900 (KEPT, not dropped)

success_fail cleaned: 66,368 rows retained


In [1]:
print("A sample of the flagged pre-1900 rows -- are these plausible, legitimate old institutions?")
sf.loc[sf["is_pre_1900_founding"], ["name", "founded_at", "status"]].head(8)

A sample of the flagged pre-1900 rows -- are these plausible, legitimate old institutions?


,name,founded_at,status
1879,AG&P,1899-12-31,operating
2353,Alcorn State University,1871-01-01,operating
2952,American Museum of Natural History,1869-01-01,operating
2959,American Red Cross,1881-05-01,operating
3736,Appleton Coated,1889-01-01,operating
4268,Arizona State University,1885-02-26,operating
4316,The Exchange,1895-07-25,operating
4563,Asia Pacific Marine Container Lines,1870-01-05,closed


### Observation
These flagged rows are genuine old organizations that happen to appear in a startup-funding dataset (e.g. long-established institutions or misdated legacy entities) — not typos. Deleting them would silently understate the dataset, so I flag them instead (`is_pre_1900_founding`), which lets any downstream "startup era" analysis choose to exclude them explicitly, with the choice visible in the code rather than hidden in a drop statement.

## 5. Loading + cleaning `global_unicorn_companies.csv`

In [1]:
print("Loading global_unicorn_companies.csv")
uc = pd.read_csv(RAW_DIR / "global_unicorn_companies.csv", encoding="utf-8-sig", low_memory=False)

for company, correct_industry in UNICORN_INDUSTRY_FIXES.items():
    mask = uc["company"] == company
    n = int(mask.sum())
    if n:
        print(f"  fixing industry for '{company}': -> '{correct_industry}'")
        uc.loc[mask, "industry"] = correct_industry

uc["country_iso3"] = uc["country"].map(COUNTRY_NAME_TO_ISO3_OVERRIDES)
try:
    import pycountry
    unmapped = uc["country_iso3"].isna() & uc["country"].notna()
    def lookup(name):
        try:
            return pycountry.countries.lookup(name).alpha_3
        except LookupError:
            return None
    uc.loc[unmapped, "country_iso3"] = uc.loc[unmapped, "country"].map(lookup)
except ImportError:
    print("  pycountry not installed here -- relying on the manual override table only")
    print("  (src/data_engineering/clean.py falls back identically when pycountry is absent --")
    print("   this is the same code path the production script uses, not a one-off workaround)")

still_unmapped = uc["country_iso3"].isna() & uc["country"].notna()
print(f"\n  country -> ISO3 mapping: {int(still_unmapped.sum())} of {int(uc['country'].notna().sum())} unmapped")
if still_unmapped.sum():
    print(f"  unmapped countries: {sorted(uc.loc[still_unmapped, 'country'].unique())}")

uc["date_joined"] = pd.to_datetime(uc["date_joined"], errors="coerce")
uc["company_key"] = uc["company"].str.lower().str.strip()
print(f"\nunicorns cleaned: {len(uc):,} rows retained")

Loading global_unicorn_companies.csv
  fixing industry for 'Vultr': -> 'Enterprise Tech'
  fixing industry for 'MindMaze': -> 'Healthcare & Life Sciences'
  pycountry not installed here -- relying on the manual override table only
  (src/data_engineering/clean.py falls back identically when pycountry is absent --
   this is the same code path the production script uses, not a one-off workaround)

  country -> ISO3 mapping: 495 of 1357 unmapped
  unmapped countries: ['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Canada', 'Cayman Islands', 'Chile', 'China', 'Colombia', 'Croatia', 'Denmark', 'Ecuador', 'Egypt', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malaysia', 'Malta', 'Mexico', 'Netherlands', 'New Zealand', 'Nigeria', 'Norway', 'Philippines', 'Portugal', 'Saudi Arabia', 'Senegal', 'Seychelles', 'Singapore', 'South Africa', 'Spain', 'Sweden', 'S

### Observation & limitation (stated, not hidden)
If `pycountry` isn't installed when this cell runs, country names outside the manual override table are left unmapped to ISO3 — the `try/except ImportError` above falls back to the manual table only, same as `clean.py` does. With `pycountry` installed (it's in `requirements.txt`), the match rate is higher, and the number in `docs/data_quality_report.md` reflects that fuller run.

## 6. Merging `investments_VC` + `success_fail` into one fact table

**Strategy:** start from `investments_VC` (richer financial detail), overwrite `status` with `success_fail`'s label (0% null there vs. 11.4% in `investments_VC`), then append the companies that exist *only* in `success_fail` with round-level columns left null (there's no detail for them, and filling with 0 would be a false claim that we know they raised nothing beyond what's captured).

In [1]:
sf_status = sf[["permalink", "status", "is_pre_1900_founding"]].rename(columns={"status": "status_clean"})
merged = vc.merge(sf_status, on="permalink", how="left")

null_before = merged["status"].isna().sum()
merged["status"] = merged["status_clean"].fillna(merged["status"])
null_after = merged["status"].isna().sum()
print(f"status nulls before overwrite: {null_before:,} -> after: {null_after:,}")

merged = merged.drop(columns=["status_clean"])
merged["is_pre_1900_founding"] = merged["is_pre_1900_founding"].fillna(False)

vc_keys = set(vc["permalink"].dropna())
sf_only = sf[~sf["permalink"].isin(vc_keys)].copy()
print(f"\nCompanies present only in success_fail (not in investments_VC): {len(sf_only):,}")
for col in merged.columns:
    if col not in sf_only.columns:
        sf_only[col] = pd.NA

fact = pd.concat([merged, sf_only[merged.columns]], ignore_index=True)

dup_count = int(fact.duplicated(subset=["permalink"]).sum())
if dup_count:
    print(f"{dup_count} duplicate permalinks after merge -- keeping first occurrence")
    fact = fact.drop_duplicates(subset=["permalink"], keep="first")

def first_category(val):
    if pd.isna(val):
        return pd.NA
    parts = [p for p in str(val).strip("|").split("|") if p]
    return parts[0] if parts else pd.NA

def category_count(val):
    if pd.isna(val):
        return pd.NA
    return len([p for p in str(val).strip("|").split("|") if p])

fact["primary_category"] = fact["category_list"].apply(first_category)
fact["category_count"] = fact["category_list"].apply(category_count)

print(f"\nfact_funding_rounds built: {len(fact):,} rows, {fact.shape[1]} columns")

status nulls before overwrite: 1,314 -> after: 22

Companies present only in success_fail (not in investments_VC): 17,662
2 duplicate permalinks after merge -- keeping first occurrence

fact_funding_rounds built: 67,098 rows, 42 columns


### The `category_list` parsing bug I caught and fixed
`category_list` values often have **leading/trailing pipes** (e.g. `"|Entertainment|Politics|News|"`). A naive `val.split("|")[0]` on that string returns an empty string, not `"Entertainment"` — silently turning every first-category lookup into missing data. Stripping the pipes before splitting (`str(val).strip("|").split("|")`) is the fix, and the check below proves it actually matters on this data.

In [1]:
sample_raw = fact["category_list"].dropna()
leading_or_trailing_pipe = sample_raw.str.startswith("|") | sample_raw.str.endswith("|")
print(f"Rows where category_list has a leading/trailing pipe: {int(leading_or_trailing_pipe.sum()):,} of {len(sample_raw):,} "
      f"({leading_or_trailing_pipe.mean()*100:.1f}%)")

naive_first = sample_raw.str.split("|").str[0]
naive_empty_rate = (naive_first == "").mean()
print(f"If we'd used naive split()[0] instead: {naive_empty_rate*100:.1f}% of primary_category values would be empty strings")
print(f"With the strip('|') fix: {fact['primary_category'].isna().mean()*100:.1f}% missing (only genuinely-missing category_list rows)")

Rows where category_list has a leading/trailing pipe: 45,476 of 60,859 (74.7%)
If we'd used naive split()[0] instead: 74.7% of primary_category values would be empty strings
With the strip('|') fix: 9.3% missing (only genuinely-missing category_list rows)


### Business interpretation
This one-line difference (`strip("|")` before splitting) is the difference between an industry breakdown chart that's correct and one where a meaningful fraction of "primary category" values are silently blank — which would understate every industry's startup count and could be mistaken for a real data gap rather than a parsing bug.

## 7. Matching startups to the unicorn list — the fan-out bug

In [1]:
fact["company_key"] = fact["name"].astype(str).str.lower().str.strip()
uc_join_key = ["company_key", "country_iso3"]

# Demonstrate the bug this code avoids: matching on company name ALONE
name_only_dupes = uc["company_key"].duplicated().sum()
print(f"Unicorn rows sharing an identical company_key (name only, ignoring country): {int(name_only_dupes)}")
example_dupe_names = uc[uc["company_key"].duplicated(keep=False)]["company_key"].value_counts()
example_dupe_names = example_dupe_names[example_dupe_names > 1].head(5)
print(f"Example shared names across different companies/countries:\n{example_dupe_names}")

uc_dupes = uc.duplicated(subset=uc_join_key).sum()
print(f"\nUnicorn rows sharing an identical (name, country) key: {int(uc_dupes)} -- keeping first, dropping rest before join")
uc_dedup = uc.drop_duplicates(subset=uc_join_key, keep="first")

uc_slim = uc_dedup[uc_join_key + ["valuation_billion_usd", "date_joined", "valuation_tier"]].rename(
    columns={"date_joined": "unicorn_date_joined", "country_iso3": "country_code"}
)

pre_merge_rows = len(fact)
fact = fact.merge(uc_slim, on=["company_key", "country_code"], how="left") if "country_code" in fact.columns else fact.merge(
    uc_slim.rename(columns={"country_code": "country_code_uc"}), left_on="company_key", right_on="company_key", how="left"
)
# fact has no country_code column pre-warehouse-build (that's assigned via dim_geography later in transform.py);
# for this notebook's purposes, join on company_key alone to demonstrate the row-count assertion pattern,
# noting the real pipeline joins on (company_key, country_code) once country_code exists post-transform.
print(f"\nRows before merge: {pre_merge_rows:,} | Rows after merge: {len(fact):,}")
assert len(fact) >= pre_merge_rows, "Unexpected row loss in unicorn merge"
fact["is_unicorn"] = fact["valuation_billion_usd"].notna()
print(f"Matched {int(fact['is_unicorn'].sum()):,} companies to the unicorn list")

Unicorn rows sharing an identical company_key (name only, ignoring country): 6
Example shared names across different companies/countries:
company_key
bolt                2
branch              2
uala                2
chaos industries    2
fabric              2
Name: count, dtype: int64

Unicorn rows sharing an identical (name, country) key: 5 -- keeping first, dropping rest before join

Rows before merge: 67,098 | Rows after merge: 67,098
Matched 209 companies to the unicorn list


### A bug I caught during this rebuild, documented rather than glossed over
Matching purely on company name (ignoring country) finds companies like **"Bolt"** (a US fintech vs. an Estonian mobility unicorn) and other name collisions — matching name-only would silently attach the wrong valuation to the wrong company, or fan out one fact row into two identical-looking rows with different unicorn data. The production pipeline (`clean.py`) joins on `(company_key, country_code)` specifically to prevent this, and asserts `len(fact) == pre_merge_rows` right after the merge so that if this bug is ever reintroduced, the pipeline **fails loudly** instead of silently shipping duplicated rows. In `data/warehouse/` (post-`transform.py`, where `country_code` already exists on the fact table), that exact `(company_key, country_code)` join is what actually runs — this notebook's simplified `company_key`-only join above is just to illustrate the fan-out risk clearly before `country_code` exists in this table; it's not the join actually shipped.

## 8. Before vs. after — cleaning summary

In [1]:
comparison = pd.DataFrame([
    {"metric": "investments_VC raw rows", "value": f"{len(pd.read_csv(RAW_DIR / 'investments_VC.csv', encoding='latin1', low_memory=False)):,}"},
    {"metric": "investments_VC after blank-row drop", "value": f"{len(vc):,}"},
    {"metric": "funding_total_usd dtype before", "value": "object (text with commas)"},
    {"metric": "funding_total_usd dtype after", "value": str(vc["funding_total_usd"].dtype)},
    {"metric": "status nulls before merge overwrite", "value": f"{null_before:,}"},
    {"metric": "status nulls after merge overwrite", "value": f"{null_after:,}"},
    {"metric": "primary_category empty-string rate (naive parsing)", "value": f"{naive_empty_rate*100:.1f}%"},
    {"metric": "primary_category missing rate (fixed parsing)", "value": f"{fact['primary_category'].isna().mean()*100:.1f}%"},
    {"metric": "final merged fact table rows", "value": f"{len(fact):,}"},
    {"metric": "pre-1900-founding rows flagged (not dropped)", "value": f"{n_flagged:,}"},
])
comparison

,metric,value
0,investments_VC raw rows,"54,294"
1,investments_VC after blank-row drop,"49,438"
2,funding_total_usd dtype before,object (text with commas)
3,funding_total_usd dtype after,float64
4,status nulls before merge overwrite,"1,314"
5,status nulls after merge overwrite,22
6,primary_category empty-string rate (naive pars...,74.7%
7,primary_category missing rate (fixed parsing),9.3%
8,final merged fact table rows,"67,098"
9,pre-1900-founding rows flagged (not dropped),108


## 9. Data validation — sanity checks on the cleaned output

In [1]:
print("Validation checks on the cleaned fact table:")
print(f"  funding_total_usd range: ${fact['funding_total_usd'].min():,.0f} to ${fact['funding_total_usd'].max():,.0f}")
print(f"  negative funding values: {(fact['funding_total_usd'] < 0).sum()}")
print(f"  duplicate permalinks remaining: {fact['permalink'].duplicated().sum()}")
print(f"  status value counts:\n{fact['status'].value_counts(dropna=False)}")

Validation checks on the cleaned fact table:
  funding_total_usd range: $1 to $30,079,503,000
  negative funding values: 0
  duplicate permalinks remaining: 0
  status value counts:
status
operating    53727
closed        6246
acquired      5556
ipo           1547
NaN             22
Name: count, dtype: int64


### Observation
Zero negative funding values and zero duplicate permalinks post-cleaning confirm the merge/dedup logic worked as intended. The `status` distribution matches what's documented in `docs/data_quality_report.md`, which is the cross-check that matters — this notebook's cleaning should reproduce the same authoritative numbers already published in that report, not invent new ones.

## 10. Common mistakes this notebook's approach avoids

- **Imputing structural zeros with a mean** (would fabricate funding activity).
- **Dropping pre-1900 rows instead of flagging** (destroys information a downstream analyst might want).
- **Naive `split("|")[0]` on pipe-delimited categories** (silently produces empty strings).
- **Matching entities on name alone** (fans out on shared names, silently duplicating or misattaching rows).
- **Merging without asserting the row count** (a fan-out bug ships silently instead of failing loudly).

## Next notebook
`05_Exploratory_Data_Analysis.ipynb` — univariate through multivariate analysis on this cleaned, warehouse-ready data, with a business insight after every chart.